In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import yfinance as yf
import math
from prophet import Prophet
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 300)

Importing plotly failed. Interactive plots will not work.


## 1. Download Data dari Yahoo Finance

In [2]:
TICKERS_MAP = {'SI=F': 'silver', 'GC=F': 'gold', 'CL=F': 'oil', 'DX-Y.NYB': 'usd'}
SERIES_LIST = ['silver', 'gold', 'oil', 'usd']
LAG_DAYS        = [1, 2, 3, 5, 7, 10, 14]
ROLLING_WINDOWS = [5, 10, 20, 30]

raw = yf.download(
    list(TICKERS_MAP.keys()),
    period='10y',
    interval='1d',
    auto_adjust=True,
    progress=False
)

close_raw = raw['Close'].copy()
close_raw = close_raw.rename(columns=TICKERS_MAP)[SERIES_LIST]
close_raw = close_raw.dropna().reset_index()
close_raw = close_raw.rename(columns={'Date': 'ds'})
close_raw['ds'] = pd.to_datetime(close_raw['ds'])
if close_raw['ds'].dt.tz is not None:
    close_raw['ds'] = close_raw['ds'].dt.tz_localize(None)

print(f'Total baris berhasil diunduh: {len(close_raw)}')
print(f'Rentang tanggal: {close_raw["ds"].min().date()} s/d {close_raw["ds"].max().date()}')
print()
close_raw[['ds','silver','gold','oil','usd']].tail(15).reset_index(drop=True)

Total baris berhasil diunduh: 2512
Rentang tanggal: 2016-05-19 s/d 2026-05-19



Ticker,ds,silver,gold,oil,usd
0,2026-04-29,71.569000,4545.200195,106.879997,98.919998
1,2026-04-30,73.533997,4614.700195,105.070000,98.080002
2,2026-05-01,75.950996,4629.899902,101.940002,98.209999
3,2026-05-04,73.071999,4519.500000,106.419998,98.470001
4,2026-05-05,73.108002,4555.799805,102.269997,98.480003
5,2026-05-06,76.810997,4681.899902,95.080002,98.019997
6,2026-05-07,79.700996,4699.799805,94.809998,98.250000
7,2026-05-08,80.394997,4720.399902,95.419998,97.839996
8,2026-05-11,85.485001,4718.700195,98.070000,97.940002
9,2026-05-12,85.129997,4677.600098,102.180000,98.290001


## 2. Seleksi Sampel Demo


In [3]:
df_all = close_raw.copy().reset_index(drop=True)
N_TEST   = max(30, int(len(df_all) * 0.2))
N_TRAIN  = len(df_all) - N_TEST

print(f'Dataset total: {len(df_all)} baris')
print(f'Train: {N_TRAIN}   Test: {N_TEST}')
print()

# Compute all non-Prophet features on full dataset
df = df_all.copy()

feat_dict = {}
for series in SERIES_LIST:
    ret = df[series].pct_change()
    for lag in LAG_DAYS:
        feat_dict[f'{series}_lag_{lag}']     = df[series].shift(lag)
        feat_dict[f'{series}_ret_lag_{lag}'] = ret.shift(lag)
    for w in ROLLING_WINDOWS:
        feat_dict[f'{series}_roll_mean_{w}'] = df[series].rolling(w).mean()
        feat_dict[f'{series}_roll_std_{w}']  = df[series].rolling(w).std()

# Technical indicators
delta    = df['silver'].diff()
up, down = delta.clip(lower=0), (-delta).clip(lower=0)
avg_up14 = up.rolling(14).mean()
avg_dn14 = down.rolling(14).mean().replace(0, 1e-9)
feat_dict['silver_rsi_14'] = (100 - 100 / (1 + avg_up14 / avg_dn14)).shift(1)

ema12 = df['silver'].ewm(span=12, adjust=False).mean()
ema26 = df['silver'].ewm(span=26, adjust=False).mean()
macd  = ema12 - ema26
feat_dict['silver_macd']        = macd.shift(1)
feat_dict['silver_macd_signal'] = macd.ewm(span=9, adjust=False).mean().shift(1)

rm20 = df['silver'].rolling(20).mean()
rs20 = df['silver'].rolling(20).std().replace(0, 1e-9)
feat_dict['silver_bb_pos'] = ((df['silver'] - rm20) / rs20).shift(1)
feat_dict['silver_atr_14'] = df['silver'].diff().abs().rolling(14).mean().shift(1)

gs = df['gold']   / df['silver'].replace(0, 1e-9)
so = df['silver'] / df['oil'].replace(0, 1e-9)
feat_dict['gold_silver_ratio_lag_1'] = gs.shift(1)
feat_dict['gold_silver_ratio_lag_5'] = gs.shift(5)
feat_dict['silver_oil_ratio_lag_1']  = so.shift(1)

feat_dict['day_of_week'] = df['ds'].dt.dayofweek
feat_dict['month']       = df['ds'].dt.month

df_feats = pd.concat([df, pd.DataFrame(feat_dict, index=df.index)], axis=1)
df_clean = df_feats.dropna().reset_index(drop=True)
df_demo  = df_clean.copy()
N_TEST   = max(30, int(len(df_clean) * 0.2))
N_TRAIN  = len(df_clean) - N_TEST

print(f'Baris setelah dropna: {len(df_clean)}')
print(f'Menampilkan 5 baris terakhir:')
print(df_demo[['ds','silver','gold','oil','usd']].tail(5).tail(10).to_string())

Dataset total: 2512 baris
Train: 2010   Test: 502

Baris setelah dropna: 2483
Menampilkan 5 baris terakhir:
             ds    silver        gold        oil       usd
2478 2026-05-13 88.888000 4697.700195 101.019997 98.480003
2479 2026-05-14 84.912003 4678.100098 101.169998 98.879997
2480 2026-05-15 77.161003 4555.799805 105.419998 99.269997
2481 2026-05-18 77.072998 4552.500000 108.660004 98.970001
2482 2026-05-19 76.800003 4561.200195 103.099998 99.133003


## 3. Feature Engineering: Lag Features

In [4]:
# Silver price lag features (demo rows only)
lag_cols_price = [f'silver_lag_{d}' for d in LAG_DAYS]
lag_cols_ret   = [f'silver_ret_lag_{d}' for d in LAG_DAYS]

lag_df = df_demo[['ds','silver'] + lag_cols_price + lag_cols_ret].copy()
print('Silver Lag Features (sampel baris data):')
print(lag_df.tail(10).to_string())
print()

# Manual verification for last row
last_full_idx = len(df_clean) - 1
print('Verifikasi baris terakhir (indeks 9 dalam demo):')
sv = df_clean['silver']
for d in [1, 2, 3]:
    price_d = df_clean['silver'].iloc[last_full_idx - d]
    price_d1 = df_clean['silver'].iloc[last_full_idx - d - 1]
    ret_d = (price_d - price_d1) / price_d1
    print(f'  silver_lag_{d}     = silver[t-{d}] = {price_d:.6f}')
    print(f'  silver_ret_lag_{d} = ({price_d:.4f} - {price_d1:.4f}) / {price_d1:.4f} = {ret_d:.8f}')

Silver Lag Features (sampel baris data):
             ds    silver  silver_lag_1  silver_lag_2  silver_lag_3  silver_lag_5  silver_lag_7  silver_lag_10  silver_lag_14  silver_ret_lag_1  silver_ret_lag_2  silver_ret_lag_3  silver_ret_lag_5  silver_ret_lag_7  silver_ret_lag_10  silver_ret_lag_14
2473 2026-05-06 76.810997     73.108002     73.071999     75.950996     71.569000     75.002998      77.892998      78.606003          0.000493         -0.037906          0.032869         -0.022348         -0.018067           0.019395          -0.011133
2474 2026-05-07 79.700996     76.810997     73.108002     73.071999     73.533997     73.205002      75.464996      81.737999          0.050651          0.000493         -0.037906          0.027456         -0.023972          -0.031171           0.039844
2475 2026-05-08 80.394997     79.700996     76.810997     73.108002     75.950996     71.569000      76.383003      79.950996          0.037625          0.050651          0.000493          0.032869

## 4. Feature Engineering: Rolling Statistics


In [5]:
roll_cols_show = ([f'silver_roll_mean_{w}' for w in ROLLING_WINDOWS]
                + [f'silver_roll_std_{w}'  for w in ROLLING_WINDOWS])

roll_df = df_demo[['ds','silver'] + roll_cols_show].copy()
print('Silver Rolling Features (sampel baris data):')
print(roll_df.tail(10).to_string())
print()

# Manual verification: roll_mean_5 for last row
last_idx = len(df_clean) - 1
w5_data = df_clean['silver'].iloc[last_idx-4:last_idx+1].values
print(f'Verifikasi silver_roll_mean_5 baris terakhir:')
print(f'  5 harga silver: {[round(x,4) for x in w5_data]}')
print(f'  mean = sum / 5 = {sum(w5_data):.6f} / 5 = {np.mean(w5_data):.6f}')
print()
w30_data = df_clean['silver'].iloc[last_idx-29:last_idx+1].values
print(f'Verifikasi silver_roll_std_30 baris terakhir (ddof=1):')
print(f'  mean_30 = {np.mean(w30_data):.6f}')
print(f'  std_30  = {np.std(w30_data, ddof=1):.6f}')

Silver Rolling Features (sampel baris data):
             ds    silver  silver_roll_mean_5  silver_roll_mean_10  silver_roll_mean_20  silver_roll_mean_30  silver_roll_std_5  silver_roll_std_10  silver_roll_std_20  silver_roll_std_30
2473 2026-05-06 76.810997           74.495198            74.410099            76.285299            74.953666           1.757555            1.740785            2.656637            3.221406
2474 2026-05-07 79.700996           75.728598            74.833699            76.456499            75.198332           2.780559            2.411993            2.764221            3.295585
2475 2026-05-08 80.394997           76.617398            75.234898            76.660049            75.622466           3.489342            2.967930            2.900481            3.106797
2476 2026-05-11 85.485001           79.099998            76.283099            77.158149            76.153799           4.581212            4.388129            3.490346            3.382399
2477 2026-05-12

## 5. Feature Engineering: RSI(14)

**Formula:**
```
delta[t]   = silver[t] - silver[t-1]
avg_up     = mean(max(delta, 0)) selama 14 periode
avg_down   = mean(max(-delta, 0)) selama 14 periode
RS         = avg_up / avg_down
RSI        = 100 - (100 / (1 + RS))
```

In [6]:
rsi_df = df_demo[['ds','silver','silver_rsi_14']].copy()
# Also compute raw (unshifted) for reference
delta_s = df_all['silver'].diff()
up_s    = delta_s.clip(lower=0)
dn_s    = (-delta_s).clip(lower=0)
rsi_raw_full = (100 - 100 / (1 + up_s.rolling(14).mean() /
                              dn_s.rolling(14).mean().replace(0,1e-9)))

print('RSI(14) sampel baris data:')
print('  silver_rsi_14 = shift(1) dari RSI')
print(rsi_df.tail(10).to_string())
print()

# Step-by-step for last row (t), before shift
last = len(df_all) - 1
win15 = df_all['silver'].iloc[last-14:last+1].values
deltas14 = np.diff(win15)
ups14    = np.where(deltas14 > 0, deltas14, 0.0)
dns14    = np.where(deltas14 < 0, -deltas14, 0.0)
avg_up_m  = np.mean(ups14)
avg_dn_m  = np.mean(dns14) if np.mean(dns14) > 0 else 1e-9
rs_m      = avg_up_m / avg_dn_m
rsi_m     = 100 - 100 / (1 + rs_m)

print(f'  15 harga silver: {[round(x,4) for x in win15]}')
print(f'  14 delta: {[round(x,6) for x in deltas14]}')
print(f'  Up changes:   {[round(x,6) for x in ups14]}')
print(f'  Down changes: {[round(x,6) for x in dns14]}')
print(f'  avg_up   = {avg_up_m:.8f}')
print(f'  avg_down = {avg_dn_m:.8f}')
print(f'  RS  = {avg_up_m:.8f} / {avg_dn_m:.8f} = {rs_m:.8f}')
print(f'  RSI = 100 - 100/(1+{rs_m:.8f}) = {rsi_m:.8f}')

RSI(14) sampel baris data:
  silver_rsi_14 = shift(1) dari RSI
             ds    silver  silver_rsi_14
2473 2026-05-06 76.810997      37.857174
2474 2026-05-07 79.700996      46.915903
2475 2026-05-08 80.394997      46.470766
2476 2026-05-11 85.485001      50.799540
2477 2026-05-12 85.129997      65.476185
2478 2026-05-13 88.888000      62.836562
2479 2026-05-14 84.912003      72.736201
2480 2026-05-15 77.161003      63.090522
2481 2026-05-18 77.072998      52.770367
2482 2026-05-19 76.800003      55.193613

  15 harga silver: [np.float64(71.569), np.float64(73.534), np.float64(75.951), np.float64(73.072), np.float64(73.108), np.float64(76.811), np.float64(79.701), np.float64(80.395), np.float64(85.485), np.float64(85.13), np.float64(88.888), np.float64(84.912), np.float64(77.161), np.float64(77.073), np.float64(76.8)]
  14 delta: [np.float64(1.964996), np.float64(2.417), np.float64(-2.878998), np.float64(0.036003), np.float64(3.702995), np.float64(2.889999), np.float64(0.694), np.flo

## 6. Feature Engineering: MACD

**Formula:**
```
alpha_12 = 2 / (12 + 1) = 0.1538
alpha_26 = 2 / (26 + 1) = 0.0741
EMA_12[t] = alpha_12 * price[t] + (1 - alpha_12) * EMA_12[t-1]
EMA_26[t] = alpha_26 * price[t] + (1 - alpha_26) * EMA_26[t-1]
MACD[t]   = EMA_12[t] - EMA_26[t]
Signal[t] = EMA_9(MACD)
```

In [7]:
ema12_full = df_all['silver'].ewm(span=12, adjust=False).mean()
ema26_full = df_all['silver'].ewm(span=26, adjust=False).mean()
macd_full  = ema12_full - ema26_full

alpha12 = 2.0 / (12 + 1)
alpha26 = 2.0 / (26 + 1)
alpha9  = 2.0 / (9 + 1)
print(f'alpha_12 = 2/(12+1) = {alpha12:.6f}')
print(f'alpha_26 = 2/(26+1) = {alpha26:.6f}')
print(f'alpha_9  = 2/(9+1)  = {alpha9:.6f}')
print()

macd_display = pd.DataFrame({
    'ds':                df_demo['ds'].values,
    'silver':            df_demo['silver'].values,
    'ema12':             ema12_full.tail(len(df_demo)).values,
    'ema26':             ema26_full.tail(len(df_demo)).values,
    'macd_raw':          macd_full.tail(len(df_demo)).values,
    'silver_macd':       df_demo['silver_macd'].values,
    'silver_macd_signal':df_demo['silver_macd_signal'].values,
})
print('MACD sampel baris data:')
print(macd_display.tail(10).to_string())
print()

# Manual for last 3 rows
last = len(df_all) - 1
print('Manual EMA untuk 3 baris terakhir:')
for i in [last-2, last-1, last]:
    sv   = df_all['silver'].iloc[i]
    e12  = ema12_full.iloc[i]
    e12p = ema12_full.iloc[i-1]
    e26  = ema26_full.iloc[i]
    e26p = ema26_full.iloc[i-1]
    print(f'  t={i}: price={sv:.4f}')
    print(f'    EMA12 = {alpha12:.4f}x{sv:.4f} + {1-alpha12:.4f}x{e12p:.4f} = {e12:.6f}')
    print(f'    EMA26 = {alpha26:.4f}x{sv:.4f} + {1-alpha26:.4f}x{e26p:.4f} = {e26:.6f}')
    print(f'    MACD  = {e12:.6f} - {e26:.6f} = {macd_full.iloc[i]:.6f}')

alpha_12 = 2/(12+1) = 0.153846
alpha_26 = 2/(26+1) = 0.074074
alpha_9  = 2/(9+1)  = 0.200000

MACD sampel baris data:
             ds    silver     ema12     ema26  macd_raw  silver_macd  silver_macd_signal
2473 2026-05-06 76.810997 74.999512 75.670841 -0.671329    -0.909478           -0.619342
2474 2026-05-07 79.700996 75.722817 75.969371 -0.246554    -0.671329           -0.629739
2475 2026-05-08 80.394997 76.441614 76.297195  0.144419    -0.246554           -0.553102
2476 2026-05-11 85.485001 77.832904 76.977773  0.855131     0.144419           -0.413598
2477 2026-05-12 85.129997 78.955534 77.581642  1.373892     0.855131           -0.159852
2478 2026-05-13 88.888000 80.483606 78.419150  2.064456     1.373892            0.146897
2479 2026-05-14 84.912003 81.164897 78.900102  2.264796     2.064456            0.530409
2480 2026-05-15 77.161003 80.548914 78.771280  1.777634     2.264796            0.877286
2481 2026-05-18 77.072998 80.014157 78.645481  1.368676     1.777634            1

## 7. Feature Engineering: Bollinger Band Position & ATR(14)

**Bollinger Band Position** posisi harga relatif terhadap pita Bollinger 20 hari:
```
BB_pos[t] = (price[t] - mean_20[t]) / std_20[t]
```

**ATR(14) Average True Range**:
```
ATR_14[t] = mean(|price[t] - price[t-1]|) untuk 14 periode terakhir
```

In [8]:
bb_atr_df = pd.DataFrame({
    'ds':            df_demo['ds'].values,
    'silver':        df_demo['silver'].values,
    'silver_bb_pos': df_demo['silver_bb_pos'].values,
    'silver_atr_14': df_demo['silver_atr_14'].values,
})
print('Bollinger Band Position & ATR(14) sampel baris data:')
print(bb_atr_df.tail(10).to_string())
print()

# Manual for last row (t), before shift
last = len(df_all) - 1
w20  = df_all['silver'].iloc[last-19:last+1].values
mean20 = np.mean(w20)
std20  = np.std(w20, ddof=1)
bb_m   = (df_all['silver'].iloc[last] - mean20) / (std20 if std20 > 0 else 1e-9)
w14d   = np.abs(np.diff(df_all['silver'].iloc[last-14:last+1].values))
atr_m  = np.mean(w14d)

print(f'  20 harga silver: {[round(x,4) for x in w20[:5]]} ... {[round(x,4) for x in w20[-5:]]}')
print(f'  mean_20 = {mean20:.6f}')
print(f'  std_20  = {std20:.6f}')
print(f'  BB_pos  = ({df_all["silver"].iloc[last]:.6f} - {mean20:.6f}) / {std20:.6f} = {bb_m:.6f}')
print(f'  14 |delta|: {[round(x,6) for x in w14d]}')
print(f'  ATR_14 = mean = {atr_m:.6f}')

Bollinger Band Position & ATR(14) sampel baris data:
             ds    silver  silver_bb_pos  silver_atr_14
2473 2026-05-06 76.810997      -1.162981       1.877356
2474 2026-05-07 79.700996       0.197881       2.078642
2475 2026-05-08 80.394997       1.173747       2.061357
2476 2026-05-11 85.485001       1.287699       1.983285
2477 2026-05-12 85.129997       2.385681       2.094000
2478 2026-05-13 88.888000       1.972550       2.013501
2479 2026-05-14 84.912003       2.360091       2.108501
2480 2026-05-15 77.161003       1.362090       2.326929
2481 2026-05-18 77.072998      -0.173651       2.782000
2482 2026-05-19 76.800003      -0.162715       2.659858

  20 harga silver: [np.float64(77.893), np.float64(75.465), np.float64(76.383), np.float64(75.003), np.float64(73.205)] ... [np.float64(88.888), np.float64(84.912), np.float64(77.161), np.float64(77.073), np.float64(76.8)]
  mean_20 = 77.876949
  std_20  = 4.815952
  BB_pos  = (76.800003 - 77.876949) / 4.815952 = -0.223621
  14 

## 8. Feature Engineering: Cross-Asset Ratios

Rasio antar aset menangkap hubungan relatif:
`gold_silver_ratio_lag_1`
`gold_silver_ratio_lag_5`
`silver_oil_ratio_lag_1`

In [9]:
ratio_df = pd.DataFrame({
    'ds':                        df_demo['ds'].values,
    'silver':                    df_demo['silver'].values,
    'gold':                      df_demo['gold'].values,
    'oil':                       df_demo['oil'].values,
    'gold_silver_ratio_lag_1':   df_demo['gold_silver_ratio_lag_1'].values,
    'gold_silver_ratio_lag_5':   df_demo['gold_silver_ratio_lag_5'].values,
    'silver_oil_ratio_lag_1':    df_demo['silver_oil_ratio_lag_1'].values,
})
print('Cross-Asset Ratio Features sampel baris data:')
print(ratio_df.tail(10).to_string())
print()

# Manual for last row
last = len(df_clean) - 1
g1  = df_clean['gold'].iloc[last-1]
s1  = df_clean['silver'].iloc[last-1]
s5  = df_clean['silver'].iloc[last-5]
g5  = df_clean['gold'].iloc[last-5]
o1  = df_clean['oil'].iloc[last-1]
print(f'Verifikasi baris terakhir:')
print(f'  gold_silver_ratio_lag_1 = gold[t-1]/silver[t-1] = {g1:.4f}/{s1:.4f} = {g1/s1:.6f}')
print(f'  gold_silver_ratio_lag_5 = gold[t-5]/silver[t-5] = {g5:.4f}/{s5:.4f} = {g5/s5:.6f}')
print(f'  silver_oil_ratio_lag_1  = silver[t-1]/oil[t-1] = {s1:.4f}/{o1:.4f} = {s1/o1:.6f}')

Cross-Asset Ratio Features sampel baris data:
             ds    silver        gold        oil  gold_silver_ratio_lag_1  gold_silver_ratio_lag_5  silver_oil_ratio_lag_1
2473 2026-05-06 76.810997 4681.899902  95.080002                62.316021                63.507946                0.714853
2474 2026-05-07 79.700996 4699.799805  94.809998                60.953510                62.756010                0.807856
2475 2026-05-08 80.394997 4720.399902  95.419998                58.967893                60.959041                0.840639
2476 2026-05-11 85.485001 4718.700195  98.070000                58.715095                61.849957                0.842538
2477 2026-05-12 85.129997 4677.600098 102.180000                55.199160                62.316021                0.871673
2478 2026-05-13 88.888000 4697.700195 101.019997                54.946555                60.953510                0.833138
2479 2026-05-14 84.912003 4678.100098 101.169998                52.849655                58.9

## 9. Feature Engineering: Calendar Features

- **`day_of_week`**: 0=Senin, 1=Selasa, ..., 4=Jumat
- **`month`**: 1–12

Fitur kalender membantu model menangkap efek hari-dalam-minggu dan musiman bulanan.

In [10]:
cal_df = pd.DataFrame({
    'ds':          df_demo['ds'].values,
    'silver':      df_demo['silver'].values,
    'day_of_week': df_demo['day_of_week'].values.astype(int),
    'day_name':    [d.strftime('%A') for d in pd.to_datetime(df_demo['ds'])],
    'month':       df_demo['month'].values.astype(int),
    'month_name':  [d.strftime('%B') for d in pd.to_datetime(df_demo['ds'])],
})
print('Calendar Features sampel baris data:')
print(cal_df.tail(10).to_string())

Calendar Features sampel baris data:
             ds    silver  day_of_week   day_name  month month_name
2473 2026-05-06 76.810997            2  Wednesday      5        May
2474 2026-05-07 79.700996            3   Thursday      5        May
2475 2026-05-08 80.394997            4     Friday      5        May
2476 2026-05-11 85.485001            0     Monday      5        May
2477 2026-05-12 85.129997            1    Tuesday      5        May
2478 2026-05-13 88.888000            2  Wednesday      5        May
2479 2026-05-14 84.912003            3   Thursday      5        May
2480 2026-05-15 77.161003            4     Friday      5        May
2481 2026-05-18 77.072998            0     Monday      5        May
2482 2026-05-19 76.800003            1    Tuesday      5        May


## 10. Prophet: Fitting & Dekomposisi Komponen


In [11]:
# Train/test split
df_train = df_demo.iloc[:N_TRAIN].copy().reset_index(drop=True)
df_test  = df_demo.iloc[N_TRAIN:].copy().reset_index(drop=True)

print(f'Train: {N_TRAIN} baris ({df_train["ds"].iloc[0].date()} s/d {df_train["ds"].iloc[-1].date()})')
print(f'Test : {N_TEST} baris  ({df_test["ds"].iloc[0].date()} s/d {df_test["ds"].iloc[-1].date()})')
print()

prophet_input = df_train[['ds','silver']].rename(columns={'silver':'y'})
print('Data input ke Prophet (y = harga silver):')
print(prophet_input.tail(10).to_string())
print()

m_prophet = Prophet(
    daily_seasonality=False,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.15,
    seasonality_mode='multiplicative',
    uncertainty_samples=0,
)
m_prophet.fit(prophet_input)
print('Prophet berhasil di fit.')

Train: 1987 baris (2016-06-30 s/d 2024-05-28)
Test : 496 baris  (2024-05-29 s/d 2026-05-19)

Data input ke Prophet (y = harga silver):
             ds         y
1977 2024-05-14 28.485001
1978 2024-05-15 29.514000
1979 2024-05-16 29.665001
1980 2024-05-17 31.047001
1981 2024-05-20 32.205002
1982 2024-05-21 31.868000
1983 2024-05-22 31.295000
1984 2024-05-23 30.284000
1985 2024-05-24 30.330000
1986 2024-05-28 31.971001



15:07:20 - cmdstanpy - INFO - Chain [1] start processing
15:07:22 - cmdstanpy - INFO - Chain [1] done processing


Prophet berhasil di fit.


In [12]:
# Predict for all 10 demo rows
ph_fc = m_prophet.predict(df_demo[['ds']].copy())
ph_comp = ph_fc[['ds','yhat','trend']].copy()
ph_comp['weekly'] = ph_fc['weekly'] if 'weekly' in ph_fc.columns else 0.0
ph_comp['yearly'] = ph_fc['yearly'] if 'yearly' in ph_fc.columns else 0.0
ph_comp.insert(1, 'silver_actual', df_demo['silver'].values)
ph_comp.columns = ['ds','silver_actual','ph_yhat','ph_trend','ph_weekly','ph_yearly']

print('Komponen Prophet untuk sampel baris data:')
print(ph_comp.tail(10).to_string())
print()
print('Catatan kolom:')
print('  ph_yhat   : prediksi Prophet')
print('  ph_trend  : komponen tren jangka panjang')
print('  ph_weekly : efek hari-dalam-minggu')
print('  ph_yearly : efek musiman tahunan')

# Add Prophet features to df_demo
df_demo['ph_yhat']   = ph_fc['yhat'].values
df_demo['ph_trend']  = ph_fc['trend'].values
df_demo['ph_weekly'] = ph_fc['weekly'].values if 'weekly' in ph_fc.columns else 0.0
df_demo['ph_yearly'] = ph_fc['yearly'].values if 'yearly' in ph_fc.columns else 0.0
print('Fitur Prophet ditambahkan ke df_demo.')

Komponen Prophet untuk sampel baris data:
             ds  silver_actual   ph_yhat  ph_trend  ph_weekly  ph_yearly
2473 2026-05-06      76.810997 31.062217  5.841884   4.063782   0.253376
2474 2026-05-07      79.700996 31.084100  5.843061   4.064176   0.255656
2475 2026-05-08      80.394997 31.111235  5.844239   4.063558   0.259845
2476 2026-05-11      85.485001 31.248990  5.847771   4.061429   0.282315
2477 2026-05-12      85.129997 31.337087  5.848948   4.065329   0.292401
2478 2026-05-13      88.888000 31.398453  5.850126   4.063782   0.303360
2479 2026-05-14      84.912003 31.474640  5.851303   4.064176   0.314906
2480 2026-05-15      77.161003 31.546640  5.852481   4.063558   0.326744
2481 2026-05-18      77.072998 31.754106  5.856013   4.061429   0.361049
2482 2026-05-19      76.800003 31.842444  5.857191   4.065329   0.371141

Catatan kolom:
  ph_yhat   : prediksi Prophet
  ph_trend  : komponen tren jangka panjang
  ph_weekly : efek hari-dalam-minggu
  ph_yearly : efek musiman t

## 11. Target: Log Return 1-Step

Model tidak memprediksi harga langsung, melainkan log return:
```
y[t] = log(silver[t] / silver[t-1])
```

In [13]:
silver_t  = df_demo['silver'].values
silver_t1 = df_demo['silver_lag_1'].values

log_ret = np.where(silver_t1 > 0, np.log(silver_t / silver_t1), 0.0)

logret_df = pd.DataFrame({
    'ds':          df_demo['ds'].values,
    'silver[t]':   silver_t,
    'silver[t-1]': silver_t1,
    'ratio':       silver_t / silver_t1,
    'log_return':  log_ret,
    'split':       ['TRAIN'] * N_TRAIN + ['TEST'] * N_TEST,
})
print('Log Return Target:')
print(logret_df.tail(10).to_string())
print()

# Manual verification
print('Verifikasi manual (rumus: log(silver[t] / silver[t-1])):')
for i in [0, 1, 7]:
    st  = silver_t[i]
    st1 = silver_t1[i]
    lr  = math.log(st / st1)
    print(f'  Baris {i}: log({st:.6f} / {st1:.6f}) = log({st/st1:.8f}) = {lr:.8f}')

y_train = log_ret[:N_TRAIN]
actual_test_prices = silver_t[N_TRAIN:]
prev_prices_test   = silver_t1[N_TRAIN:]

print()
print(f'y_train (8 log returns): {[round(x,8) for x in y_train]}')
print(f'Harga aktual test:       {[round(x,4) for x in actual_test_prices]}')
print(f'Harga sebelumnya (t-1):  {[round(x,4) for x in prev_prices_test]}')

Log Return Target:
             ds  silver[t]  silver[t-1]    ratio  log_return split
2473 2026-05-06  76.810997    73.108002 1.050651    0.049410  TEST
2474 2026-05-07  79.700996    76.810997 1.037625    0.036934  TEST
2475 2026-05-08  80.394997    79.700996 1.008708    0.008670  TEST
2476 2026-05-11  85.485001    80.394997 1.063312    0.061389  TEST
2477 2026-05-12  85.129997    85.485001 0.995847   -0.004161  TEST
2478 2026-05-13  88.888000    85.129997 1.044144    0.043198  TEST
2479 2026-05-14  84.912003    88.888000 0.955270   -0.045762  TEST
2480 2026-05-15  77.161003    84.912003 0.908717   -0.095721  TEST
2481 2026-05-18  77.072998    77.161003 0.998859   -0.001141  TEST
2482 2026-05-19  76.800003    77.072998 0.996458   -0.003548  TEST

Verifikasi manual (rumus: log(silver[t] / silver[t-1])):
  Baris 0: log(18.582001 / 18.362000) = log(1.01198133) = 0.01191012
  Baris 1: log(19.544001 / 18.582001) = log(1.05177052) = 0.05047496
  Baris 7: log(20.129999 / 20.264000) = log(0.99

## 12. Matriks Fitur Final (X)

In [14]:
price_lag_cols = [f'{s}_lag_{d}'     for s in SERIES_LIST for d in LAG_DAYS]
ret_lag_cols   = [f'{s}_ret_lag_{d}' for s in SERIES_LIST for d in LAG_DAYS]
roll_cols_all  = [f'{s}_{fn}_{w}'
                  for s in SERIES_LIST
                  for fn in ['roll_mean','roll_std']
                  for w in ROLLING_WINDOWS]
calendar_cols  = ['day_of_week', 'month']
prophet_cols   = ['ph_yhat', 'ph_trend', 'ph_weekly', 'ph_yearly']
tech_cols      = ['silver_rsi_14','silver_macd','silver_macd_signal','silver_bb_pos','silver_atr_14']
ratio_cols     = ['gold_silver_ratio_lag_1','gold_silver_ratio_lag_5','silver_oil_ratio_lag_1']
feature_cols   = price_lag_cols + ret_lag_cols + roll_cols_all + calendar_cols + prophet_cols + tech_cols + ratio_cols

print(f'Total fitur: {len(feature_cols)}')
print()

X_all   = df_demo[feature_cols].values
X_train = X_all[:N_TRAIN]
X_test  = X_all[N_TRAIN:]

print(f'X_train shape: {X_train.shape}  (8 baris × 102 fitur)')
print(f'X_test  shape: {X_test.shape}  (2 baris × 102 fitur)')
print()

# Show first 10 features of X_train as DataFrame
X_train_show = pd.DataFrame(X_train[:, :10], columns=feature_cols[:10])
X_train_show.insert(0, 'ds', df_demo['ds'].iloc[:N_TRAIN].values)
print('X_train — 10 fitur pertama (dari 102):')
print(X_train_show.tail(10).to_string())
print()
# Show last 10 features
X_train_show2 = pd.DataFrame(X_train[:, -10:], columns=feature_cols[-10:])
X_train_show2.insert(0, 'ds', df_demo['ds'].iloc[:N_TRAIN].values)
print('X_train — 10 fitur terakhir (dari 102):')
print(X_train_show2.tail(10).to_string())

Total fitur: 102

X_train shape: (1987, 102)  (8 baris × 102 fitur)
X_test  shape: (496, 102)  (2 baris × 102 fitur)

X_train — 10 fitur pertama (dari 102):
             ds  silver_lag_1  silver_lag_2  silver_lag_3  silver_lag_5  silver_lag_7  silver_lag_10  silver_lag_14  gold_lag_1  gold_lag_2  gold_lag_3
1977 2024-05-14     28.221001     28.275000     28.132000     27.302999     26.445000      26.391001      27.323999 2336.100098 2367.300049 2332.100098
1978 2024-05-15     28.485001     28.221001     28.275000     27.361000     27.368999      26.489000      27.341999 2353.399902 2336.100098 2367.300049
1979 2024-05-16     29.514000     28.485001     28.221001     28.132000     27.302999      26.583000      27.240999 2388.699951 2353.399902 2336.100098
1980 2024-05-17     29.665001     29.514000     28.485001     28.275000     27.361000      26.445000      27.372999 2380.000000 2388.699951 2353.399902
1981 2024-05-20     31.047001     29.665001     29.514000     28.221001     28.1320

## 13. Training & Prediksi XGBoost


In [15]:
xgb_model = XGBRegressor(
    n_estimators=1000, learning_rate=0.01, max_depth=3,
    subsample=0.7, colsample_bytree=0.6,
    reg_alpha=0.5, reg_lambda=2.0, min_child_weight=5,
    random_state=42, verbosity=0,
)
xgb_model.fit(X_train, y_train)
print()

import sys
import os
sys.path.append(os.path.join(os.getcwd(), 'backend'))
from models.forecaster import _feature_row

xgb_test_preds = []
xgb_log_preds  = []
silver_hist = list(df_clean['silver'].values[:-N_TEST])
gold_hist   = list(df_clean['gold'].values)
oil_hist    = list(df_clean['oil'].values)
usd_hist    = list(df_clean['usd'].values)
n_total     = len(df_clean)

for i in range(N_TEST):
    cutoff = n_total - N_TEST + i
    fdate  = df_test['ds'].iloc[i]
    ph_row = df_demo[df_demo['ds'] == fdate].iloc[0]
    x = _feature_row(
        silver_hist, gold_hist[:cutoff], oil_hist[:cutoff], usd_hist[:cutoff],
        float(ph_row['ph_yhat']), float(ph_row['ph_trend']),
        float(ph_row['ph_weekly']), float(ph_row['ph_yearly']),
        fdate, feature_cols
    )
    prev_price = silver_hist[-1]
    log_ret_pred = float(xgb_model.predict(x.reshape(1, -1))[0])
    price_pred = prev_price * np.exp(log_ret_pred)
    xgb_test_preds.append(price_pred)
    xgb_log_preds.append(log_ret_pred)
    silver_hist.append(float(actual_test_prices[i]))

xgb_price_pred = np.array(xgb_test_preds)
xgb_log_pred   = np.array(xgb_log_preds)

## 14. Training & Prediksi LightGBM


In [16]:
lgbm_model = LGBMRegressor(
    n_estimators=1500, learning_rate=0.005, max_depth=3, num_leaves=7,
    subsample=0.7, subsample_freq=5, colsample_bytree=0.6,
    reg_alpha=0.5, reg_lambda=3.0, min_child_samples=30,
    random_state=42, verbose=-1,
)
lgbm_model.fit(X_train, y_train)
print()

lgbm_test_preds = []
lgbm_log_preds  = []
silver_hist = list(df_clean['silver'].values[:-N_TEST])

for i in range(N_TEST):
    cutoff = n_total - N_TEST + i
    fdate  = df_test['ds'].iloc[i]
    ph_row = df_demo[df_demo['ds'] == fdate].iloc[0]
    x = _feature_row(
        silver_hist, gold_hist[:cutoff], oil_hist[:cutoff], usd_hist[:cutoff],
        float(ph_row['ph_yhat']), float(ph_row['ph_trend']),
        float(ph_row['ph_weekly']), float(ph_row['ph_yearly']),
        fdate, feature_cols
    )
    prev_price = silver_hist[-1]
    log_ret_pred = float(lgbm_model.predict(x.reshape(1, -1))[0])
    price_pred = prev_price * np.exp(log_ret_pred)
    lgbm_test_preds.append(price_pred)
    lgbm_log_preds.append(log_ret_pred)
    silver_hist.append(float(actual_test_prices[i]))

lgbm_price_pred = np.array(lgbm_test_preds)
lgbm_log_pred   = np.array(lgbm_log_preds)

## 15. Evaluasi Metrik


In [17]:
actual   = actual_test_prices
n        = len(actual)

print('PERBANDINGAN PREDIKSI VS AKTUAL')
comp = pd.DataFrame({
    'Tanggal':    [str(d.date()) for d in df_test['ds']],
    'Aktual':     actual,
    'XGBoost':    xgb_price_pred,
    'LightGBM':   lgbm_price_pred,
    'Err_XGB':    actual - xgb_price_pred,
    'Err_LGBM':   actual - lgbm_price_pred,
})
print(comp.to_string(index=False))

# XGBoost metrics step-by-step
print()
print('=' * 65)
print('METRIK XGBOOST')
print('=' * 65)
xgb_sq   = (actual - xgb_price_pred) ** 2
xgb_abs  = np.abs(actual - xgb_price_pred)
safe_actual = np.where(actual == 0, 1e-9, actual)
xgb_pct  = xgb_abs / safe_actual * 100

print(f'n = {n}')
print()
print('Squared Errors:')
for i in range(n):
    print(f'  ({actual[i]:.4f} - {xgb_price_pred[i]:.4f})^2 = {actual[i]-xgb_price_pred[i]:+.6f}^2 = {xgb_sq[i]:.8f}')
mse_xgb  = np.mean(xgb_sq)
rmse_xgb = np.sqrt(mse_xgb)
print(f'  MSE  = ({" + ".join([f"{x:.8f}" for x in xgb_sq])}) / {n} = {mse_xgb:.8f}')
print(f'  RMSE = sqrt({mse_xgb:.8f}) = {rmse_xgb:.8f}')
print()
print('Absolute Errors:')
for i in range(n):
    print(f'  |{actual[i]:.4f} - {xgb_price_pred[i]:.4f}| = {xgb_abs[i]:.8f}')
mae_xgb = np.mean(xgb_abs)
print(f'  MAE = ({" + ".join([f"{x:.8f}" for x in xgb_abs])}) / {n} = {mae_xgb:.8f}')
print()
print('Percentage Errors:')
for i in range(n):
    print(f'  |{actual[i]:.4f} - {xgb_price_pred[i]:.4f}| / {actual[i]:.4f} x100 = {xgb_pct[i]:.6f}%')
mape_xgb = np.mean(xgb_pct)
print(f'  MAPE = {mape_xgb:.6f}%')
print()
mean_actual = np.mean(actual)
ss_res_xgb = np.sum((actual - xgb_price_pred)**2)
ss_tot     = np.sum((actual - mean_actual)**2)
r2_xgb = (1 - ss_res_xgb / ss_tot) if ss_tot > 0 else float('nan')
print(f'R2 = 1 - SS_res / SS_tot')
print(f'   SS_res = {ss_res_xgb:.8f}')
print(f'   SS_tot = sum((actual - mean_actual)^2), mean_actual = {mean_actual:.6f}')
for i in range(n):
    print(f'     ({actual[i]:.4f} - {mean_actual:.4f})^2 = {(actual[i]-mean_actual)**2:.8f}')
print(f'   SS_tot = {ss_tot:.8f}')
print(f'   R2  = 1 - {ss_res_xgb:.8f} / {ss_tot:.8f} = {r2_xgb:.8f}')

# LightGBM metrics
print()

print('METRIK LIGHTGBM')

lgbm_sq  = (actual - lgbm_price_pred) ** 2
lgbm_abs = np.abs(actual - lgbm_price_pred)
safe_actual = np.where(actual == 0, 1e-9, actual)
lgbm_pct = lgbm_abs / safe_actual * 100
mse_lgbm  = np.mean(lgbm_sq)
rmse_lgbm = np.sqrt(mse_lgbm)
mae_lgbm  = np.mean(lgbm_abs)
mape_lgbm = np.mean(lgbm_pct)
ss_res_lgbm = np.sum((actual - lgbm_price_pred)**2)
r2_lgbm = (1 - ss_res_lgbm / ss_tot) if ss_tot > 0 else float('nan')

print(f'n = {n}')
print('Squared Errors:')
for i in range(n):
    print(f'  ({actual[i]:.4f} - {lgbm_price_pred[i]:.4f})^2 = {actual[i]-lgbm_price_pred[i]:+.6f}^2 = {lgbm_sq[i]:.8f}')
print(f'  RMSE = sqrt({mse_lgbm:.8f}) = {rmse_lgbm:.8f}')
print(f'  MAE  = {mae_lgbm:.8f}')
print(f'  MAPE = {mape_lgbm:.6f}%')
print(f'  R2   = 1 - {ss_res_lgbm:.8f} / {ss_tot:.8f} = {r2_lgbm:.8f}')

PERBANDINGAN PREDIKSI VS AKTUAL
   Tanggal     Aktual    XGBoost   LightGBM    Err_XGB   Err_LGBM
2024-05-29  32.202999  31.987425  31.964727   0.215575   0.238272
2024-05-30  31.388000  32.120608  32.103033  -0.732608  -0.715032
2024-05-31  30.297001  31.535688  31.460404  -1.238687  -1.163403
2024-06-03  30.641001  30.336759  30.318002   0.304242   0.322999
2024-06-04  29.488001  30.535081  30.518469  -1.047080  -1.030469
2024-06-05  29.948000  29.544857  29.517626   0.403143   0.430374
2024-06-06  31.247000  29.921671  29.928724   1.325329   1.318275
2024-06-07  29.334999  31.195022  31.183051  -1.860023  -1.848052
2024-06-10  29.768999  29.507083  29.486777   0.261916   0.282222
2024-06-11  29.132999  29.774549  29.748188  -0.641550  -0.615189
2024-06-12  30.176001  29.171942  29.178503   1.004059   0.997498
2024-06-13  28.990000  30.182982  30.150772  -1.192982  -1.160772
2024-06-14  29.402000  29.032700  29.043461   0.369300   0.358540
2024-06-17  29.325001  29.293802  29.317706 

## 16. Ringkasan Akhir


In [18]:
print('RINGKASAN AKHIR')

print()
print('Harga Prediksi vs Aktual (USD/troy oz):')
summary = pd.DataFrame({
    'Tanggal':  [str(d.date()) for d in df_test['ds']],
    'Aktual':   [f'${x:.4f}' for x in actual],
    'XGBoost':  [f'${x:.4f}' for x in xgb_price_pred],
    'LightGBM': [f'${x:.4f}' for x in lgbm_price_pred],
    'Err_XGB%': [f'{abs(actual[i]-xgb_price_pred[i])/actual[i]*100:.4f}%' for i in range(n)],
    'Err_LGBM%':[f'{abs(actual[i]-lgbm_price_pred[i])/actual[i]*100:.4f}%' for i in range(n)],
})
print('... Menampilkan 15 baris terakhir ...')
print(summary.tail(15).to_string(index=False))
print()

print('Metrik Evaluasi:')
metrics = pd.DataFrame({
    'Metrik':   ['RMSE (USD)', 'MAE (USD)', 'MAPE (%)', 'R2'],
    'XGBoost':  [f'{rmse_xgb:.6f}',  f'{mae_xgb:.6f}',  f'{mape_xgb:.4f}',  f'{r2_xgb:.6f}'],
    'LightGBM': [f'{rmse_lgbm:.6f}', f'{mae_lgbm:.6f}', f'{mape_lgbm:.4f}', f'{r2_lgbm:.6f}'],
})
print(metrics.to_string(index=False))
print()

RINGKASAN AKHIR

Harga Prediksi vs Aktual (USD/troy oz):
... Menampilkan 15 baris terakhir ...
   Tanggal   Aktual  XGBoost LightGBM Err_XGB% Err_LGBM%
2026-04-29 $71.5690 $73.0590 $73.1537  2.0819%   2.2142%
2026-04-30 $73.5340 $71.1868 $71.3060  3.1920%   3.0299%
2026-05-01 $75.9510 $72.7843 $72.8901  4.1694%   4.0300%
2026-05-04 $73.0720 $75.6767 $75.6890  3.5646%   3.5814%
2026-05-05 $73.1080 $73.0002 $73.0071  0.1475%   0.1380%
2026-05-06 $76.8110 $72.5957 $72.7610  5.4879%   5.2727%
2026-05-07 $79.7010 $76.6605 $76.6805  3.8149%   3.7897%
2026-05-08 $80.3950 $79.5531 $79.4738  1.0472%   1.1459%
2026-05-11 $85.4850 $80.1162 $80.1371  6.2804%   6.2559%
2026-05-12 $85.1300 $85.5112 $85.4774  0.4477%   0.4081%
2026-05-13 $88.8880 $85.2919 $85.1939  4.0457%   4.1559%
2026-05-14 $84.9120 $88.2025 $88.2340  3.8752%   3.9123%
2026-05-15 $77.1610 $84.6367 $84.6204  9.6885%   9.6674%
2026-05-18 $77.0730 $76.9279 $76.9640  0.1882%   0.1415%
2026-05-19 $76.8000 $76.7113 $76.7636  0.1155%   0